In [1]:
import os

In [2]:
%pwd

'd:\\ThirdYear\\ImageProcessing\\Kindy-disease-Classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\ThirdYear\\ImageProcessing\\Kindy-disease-Classification'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [19]:
import os
import zipfile
import gdown
from pathlib import Path
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size
from cnnClassifier.entity.config_entity import DataIngestionConfig


class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs(os.path.dirname(zip_download_dir), exist_ok=True)
            
            # تنزيل الملف المباشر من جوجل درايف
            gdown.download(dataset_url, zip_download_dir, quiet=False)
            logger.info(f"Downloaded data into {zip_download_dir}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [20]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-10-14 18:37:36,163: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-14 18:37:36,163: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-14 18:37:36,163: INFO: common: created directory at: artifacts]
[2026-10-14 18:37:36,163: INFO: common: created directory at: artifacts/data_ingestion]


Downloading...
From (original): https://drive.google.com/uc?id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3
From (redirected): https://drive.google.com/uc?id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3&confirm=t&uuid=e9f7fa41-23a4-4bd6-b3d7-1c3ab2abed5c
To: d:\ThirdYear\ImageProcessing\Kindy-disease-Classification\artifacts\data_ingestion\data.zip
100%|██████████| 57.7M/57.7M [02:19<00:00, 413kB/s]

[2026-10-14 18:39:59,077: INFO: 3076173673: Downloaded data into artifacts/data_ingestion/data.zip]
